In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

In [4]:
# Datasets and DataLoader
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image -> scale -> normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./data", train= True, download=True, transform=transform)
testset = CIFAR10(root="./data", train= False, download=True, transform=transform)

In [7]:
train_loader = DataLoader(trainset, batch_size=64, shuffle=True)
test_loader = DataLoader(testset, batch_size=64)

### Building the CNN

In [11]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            # first convolution layer
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Kernel size = 2 & stride = 2

            # second convolution layer
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # third convolution layer
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # Flattening
        x = self.fc_layers(x)

        return x

In [13]:
model = CNN()

In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN

In [16]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()

        output = model.forward(images) # FP
        loss = criterion(output, labels) # Loss function
        loss.backward() # BP
        optimizer.step() # Update parameters

        epoch_training_loss += loss.item()

    print(f"Epoch = {epoch + 1}/{epochs} & Loss = {epoch_training_loss/len(train_loader)}")

Epoch = 1/10 & Loss = 1.3788409483859607
Epoch = 2/10 & Loss = 0.9543957634807547
Epoch = 3/10 & Loss = 0.7674480395777451
Epoch = 4/10 & Loss = 0.6381934784791049
Epoch = 5/10 & Loss = 0.5258096878790794
Epoch = 6/10 & Loss = 0.43222579392402066
Epoch = 7/10 & Loss = 0.34674180105633445
Epoch = 8/10 & Loss = 0.27078043169263377
Epoch = 9/10 & Loss = 0.20864944935054577
Epoch = 10/10 & Loss = 0.16875642857483358


In [ ]:
# Training with validation
train_loss = []
val_loss = []

epochs = 10

for epoch in range(epochs):
    # Training
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()

        output = model.forward(images) # FP
        loss = criterion(output, labels) # Loss function
        loss.backward() # BP
        optimizer.step() # Update parameters

        running_loss += loss.item()

    epoch_train_loss = running_loss / len(train_loader)
    train_loss.append(epoch_train_loss)

    # Validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model.forward(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss

    epoch_val_loss = running_val_loss/len(test_loader)
    val_loss.append(epoch_val_loss)

    print(f"For epoch:{epoch+1}/{epochs} ==> training loss = {epoch_train_loss} & validation loss = {epoch_val_loss}")

In [ ]:
# Visualise the losses
import matplotlib.pyplot as plt
import pandas as pd

loss_df = pd.DataFrame({
    "Training loss" : train_loss,
    "Validation loss" : val_loss
})

plt.figure(figsize=(12, 8))
plt.plot(loss_df["Training loss"], label="Training Loss")
plt.plot(loss_df["Validation loss"], label="Validation Loss")

plt.xlabel("Epochs")
plt.ylabel("Losses")

plt.legend()

In [17]:
# Evaluation
correct_preds = 0
total_labels = 0

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_preds += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {(correct_preds/total_labels) * 100} %")

Accuracy = 74.87 %
